# DQI Reed–Solomon / Berlekamp–Massey reproduction

This notebook reproduces the published `dqi-reed-solomon` discovered circuit and evaluator from [`andigu/uncompute`](https://github.com/andigu/uncompute), pinned to commit `64b5a13aa249f1360846657392e6c66817b26d0f`.

It is a **baseline reproduction**, not a new optimization. The upstream task uses 70 syndrome symbols over GF(2^12) and evaluates the reversible decoder against its stated input distribution.

Expected certified resource metrics at the pinned revision: **442,397 Toffoli gates**, **1,837 peak live qubits**, **TQ-cost 812,683,289**. The evaluator also checks the phase, syndrome preservation, and workspace cleanup.

The Kaggle kernel requests **GPU T4 ×2**. The upstream circuit builder and resource evaluator are CPU-oriented; the GPU is provisioned and reported below but is not used to accelerate the evaluator. This avoids falsely treating GPU allocation as a speedup.

The upstream repository is licensed Apache-2.0; this notebook fetches the source at a pinned commit instead of copying its implementation.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version)
gpu = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
if gpu.returncode == 0:
    print(gpu.stdout)
else:
    print('nvidia-smi unavailable; continuing because the reference evaluator is CPU-oriented.')

In [ ]:
UPSTREAM_URL = 'https://github.com/andigu/uncompute.git'
UPSTREAM_COMMIT = '64b5a13aa249f1360846657392e6c66817b26d0f'
UPSTREAM = WORK / 'uncompute-upstream'

if UPSTREAM.exists():
    shutil.rmtree(UPSTREAM)
subprocess.run(['git', 'init', str(UPSTREAM)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'remote', 'add', 'origin', UPSTREAM_URL], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'fetch', '--depth=1', 'origin', UPSTREAM_COMMIT], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(UPSTREAM), 'rev-parse', 'HEAD'], text=True).strip()
print('Pinned upstream commit:', actual_commit)
assert actual_commit == UPSTREAM_COMMIT, (actual_commit, UPSTREAM_COMMIT)
assert (UPSTREAM / 'tasks/dqi-reed-solomon/circuits/discovered/circuit.py').is_file()
assert (UPSTREAM / 'tools/eval.py').is_file()
print('Located the BM circuit and official evaluator.')

In [ ]:
# Install the upstream project's locked Python environment and optional native evaluator kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.12'], check=True)
subprocess.run(['uv', 'sync', '--locked'], cwd=UPSTREAM, check=True)
# The native C kernel accelerates evaluation; the pure-Python evaluator remains the reference path.
subprocess.run(['make', 'kernel'], cwd=UPSTREAM, check=True)
print('Environment ready.')

In [ ]:
RESULTS = WORK / 'bm-reproduction-results'
RESULTS.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env['UNCOMPUTE_KERNEL'] = '1'
cmd = [
    'uv', 'run', 'python', 'tools/eval.py', 'dqi-reed-solomon',
    '--json', '--out', str(RESULTS), '--jobs', '4', '--timeout', '3600'
]
print('Running:', ' '.join(cmd))
run = subprocess.run(cmd, cwd=UPSTREAM, env=env, text=True, capture_output=True)
print('Exit code:', run.returncode)
if run.stdout:
    print('STDOUT (tail):\n', run.stdout[-6000:])
if run.stderr:
    print('STDERR (tail):\n', run.stderr[-6000:])
if run.returncode != 0:
    raise RuntimeError('Upstream evaluator failed; inspect the stdout/stderr above.')

report_path = RESULTS / 'report.json'
if report_path.exists():
    report = json.loads(report_path.read_text())
else:
    report = json.loads(run.stdout.strip().splitlines()[-1])
metrics = report.get('metrics', {})
summary = {
    'upstream_url': UPSTREAM_URL,
    'upstream_commit': UPSTREAM_COMMIT,
    'task': 'dqi-reed-solomon',
    'verdict': report.get('verdict', report.get('status')),
    'toffoli': metrics.get('toffoli'),
    'peak_qubits': metrics.get('qubits'),
    'tq_cost': metrics.get('tq_cost'),
    'metrics': metrics,
}
(RESULTS / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))

expected = {'toffoli': 442397, 'peak_qubits': 1837, 'tq_cost': 812683289}
assert summary['verdict'] == 'OK', f"Unexpected evaluator verdict: {summary['verdict']}"
for key, value in expected.items():
    assert summary[key] == value, f"{key}: expected {value}, got {summary[key]}"
print('PASS: reproduced the published BM circuit resource metrics.')